In [1]:
!pwd

/home/alan_khang/dev/mirror3d


In [1]:
import os
import datetime

import sys, os

root = os.getcwd()
sys.path.append(f"{root}/mirror3d/mirror3dnet")

from mirror3d.utils import RefineDepth, read_json, unit_vector
from mirror3d_lib.engine.defaults import Mirror3dTrainer
from mirror3d_lib.config.config import get_cfg
from mirror3d_lib.data.datasets.register_mirror3d_coco import register_mirror3d_coco_instances

from detectron2.checkpoint import DetectionCheckpointer
from detectron2.evaluation import inference_context
from detectron2.utils.visualizer import Visualizer
from detectron2.utils.visualizer import ColorMode
from detectron2.data import MetadataCatalog
from detectron2.engine.defaults import DefaultPredictor
from detectron2.modeling import build_model

import torch
import torch_tensorrt
import numpy as np
import cv2
import matplotlib.pyplot as plt
import shutil

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
os.environ["ROBOTFLOW_VERSION"] = "11"
os.environ["DATADIR"] = f"./data/Mirror-Glass_Segmentation.v{os.environ['ROBOTFLOW_VERSION']}i.coco-segmentation"
os.environ["DATE"] = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M")

**Export to ONNX**

In [3]:
cmd = f"python3 tools/export_2_onnx.py \
    --config-file mirror3d/mirror3dnet/config/mirror3dnet_config.yml \
    --export-method tracing \
    --format onnx \
    --output ./output/deploy/{os.environ['DATE']} \
    --data-dir {os.environ['DATADIR']} \
    --model-weights ./output/mirror_glass_segm_v11/m3n_full_rawD_resume_2026-03-28-19-07-19/model_final.pth"

os.system(cmd)

[03/30 15:52:49 detectron2]: Command line arguments: Namespace(format='onnx', export_method='tracing', data_dir='./data/Mirror-Glass_Segmentation.v11i.coco-segmentation', model_weights='./output/mirror_glass_segm_v11/m3n_full_rawD_resume_2026-03-28-19-07-19/model_final.pth', config_file='mirror3d/mirror3dnet/config/mirror3dnet_config.yml', sample_image=None, run_eval=False, output='./output/deploy/2026-03-30-15-52', opts=[])


[W init.cpp:855] Warning: Use _jit_set_fusion_strategy, bailout depth is deprecated. Setting to (STATIC, 1) (function operator())


[03/30 15:52:50 d2.engine.defaults]: Model:
Mirror3d_GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
 

The checkpoint state_dict contains keys that are not used by the model:
  depth_predictor.conv1.0.{bias, weight}
  depth_predictor.conv1.1.{bias, weight}
  depth_predictor.conv2.0.{bias, weight}
  depth_predictor.conv2.1.{bias, weight}
  depth_predictor.conv3.0.{bias, weight}
  depth_predictor.conv3.1.{bias, weight}
  depth_predictor.conv4.0.{bias, weight}
  depth_predictor.conv4.1.{bias, weight}
  depth_predictor.conv5.0.{bias, weight}
  depth_predictor.conv5.1.{bias, weight}
  depth_predictor.deconv1.1.{bias, weight}
  depth_predictor.deconv1.2.{bias, weight}
  depth_predictor.deconv2.1.{bias, weight}
  depth_predictor.deconv2.2.{bias, weight}
  depth_predictor.deconv3.1.{bias, weight}
  depth_predictor.deconv3.2.{bias, weight}
  depth_predictor.deconv4.1.{bias, weight}
  depth_predictor.deconv4.2.{bias, weight}
  depth_predictor.deconv5.1.{bias, weight}
  depth_predictor.deconv5.2.{bias, weight}
  depth_predictor.depth_pred.{bias, weight}
/home/alan_khang/dev/detectron2/detectron2/s

[03/30 15:52:50 d2.data.build]: Distribution of instances among all 1 categories:
|   category   | #instances   |
|:------------:|:-------------|
| mirror_glass | 120          |
|              |              |
[03/30 15:52:50 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[03/30 15:52:50 d2.data.common]: Serializing 65 elements to byte tensors and concatenating them all ...
[03/30 15:52:50 d2.data.common]: Serialized dataset takes 0.11 MiB


/home/alan_khang/miniconda3/envs/mirror3d/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/alan_khang/dev/detectron2/detectron2/structures/boxes.py:151: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if tensor.numel() == 0:
/home/alan_khang/dev/detectron2/detectron2/structures/boxes.py:155: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace mig

[03/30 15:52:56 detectron2]: Inputs schema: TupleSchema(schemas=[ListSchema(schemas=[DictSchema(schemas=[IdentitySchema()], sizes=[1], keys=dict_keys(['image']))], sizes=[1])], sizes=[1])
[03/30 15:52:56 detectron2]: Outputs schema: ListSchema(schemas=[DictSchema(schemas=[ListSchema(schemas=[InstancesSchema(schemas=[TensorWrapSchema(class_name='detectron2.structures.Boxes'), IdentitySchema(), IdentitySchema(), IdentitySchema(), IdentitySchema(), IdentitySchema(), IdentitySchema()], sizes=[1, 1, 1, 1, 1, 1, 1], keys=dict_keys(['pred_boxes', 'scores', 'anchor_scores', 'pred_anchor_classes', 'pred_residuals', 'pred_classes', 'pred_masks']))], sizes=[8])], sizes=[8], keys=dict_keys(['instances']))], sizes=[8])
[03/30 15:52:56 detectron2]: Success.


0

**Create ONNX Graph**

In [4]:
cmd = f"python3 tools/create_onnx.py \
    --exported_onnx ./output/deploy/{os.environ['DATE']}/model.onnx \
    --onnx ./output/deploy/{os.environ['DATE']}/converted.onnx \
    --det2_config mirror3d/mirror3dnet/config/mirror3dnet_config.yml \
    --det2_weights ./output/mirror_glass_segm_v11/m3n_full_rawD_resume_2026-03-28-19-07-19/model_final.pth \
    --sample_image ./data/Mirror-Glass_Segmentation.v11i.coco-segmentation/valid/oak_front_frame_0000139_2026_03_23-20-44-59_jpg.rf.713bb2f4f5d465737f1d55b499f2e755.jpg \
    --first_nms_threshold 0.7 \
    --second_nms_threshold 0.7 \
    --data_dir {os.environ['DATADIR']}"

os.system(cmd)

INFO:ModelHelper:ONNX graph loaded successfully
2026-03-30 15:53:05.990424931 [W:onnxruntime:, unsqueeze_elimination.cc:23 Apply] UnsqueezeElimination cannot remove node Unsqueeze_2265
2026-03-30 15:53:05.990439009 [W:onnxruntime:, unsqueeze_elimination.cc:23 Apply] UnsqueezeElimination cannot remove node Unsqueeze_2262
2026-03-30 15:53:05.995890749 [W:onnxruntime:, unsqueeze_elimination.cc:23 Apply] UnsqueezeElimination cannot remove node Unsqueeze_3368
2026-03-30 15:53:05.995903886 [W:onnxruntime:, unsqueeze_elimination.cc:23 Apply] UnsqueezeElimination cannot remove node Unsqueeze_3365
INFO:ModelHelper:Number of FPN output channels is 256
INFO:ModelHelper:Number of classes is 1
INFO:ModelHelper:First NMS max proposals is 50
INFO:ModelHelper:First NMS iou threshold is 0.5
INFO:ModelHelper:First NMS score threshold is 0.01
INFO:ModelHelper:First ROIAlign type is ROIAlignV2
INFO:ModelHelper:First ROIAlign pooled size is 7
INFO:ModelHelper:First ROIAlign sampling ratio is 0
INFO:ModelHe

0

**Build TensorRT Engine**

In [6]:
cmd = f"python3 tools/build_engine.py \
    --onnx ./output/deploy/{os.environ['DATE']}/converted.onnx \
    --engine ./output/deploy/{os.environ['DATE']}/engine.trt \
    --precision fp16 \
    --verbose"

os.system(cmd)

<frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cuda module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.driver module instead.


[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::BatchedNMSDynamic_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::BatchedNMS_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::BatchTilePlugin_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::Clip_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::CoordConvAC version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::CropAndResizeDynamic version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::CropAndResize version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::DecodeBbox3DPlugin version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::DetectionLayer_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::EfficientNMS_Explicit_TF_TRT version 1
[03/30/2026-15:48:15] [TRT] [V] Registered plugin creator - ::EfficientNMS_Implicit_TF_TRT version 1

/home/alan_khang/dev/mirror3d/tools/build_engine.py:134: DeprecationWarning: Use set_memory_pool_limit instead.
  self.config.max_workspace_size = workspace * (2 ** 30)


-15:48:20] [TRT] [V] /proposal_generator/anchor_generator/Constant_48 [Constant] inputs: 
[03/30/2026-15:48:20] [TRT] [V] /proposal_generator/anchor_generator/Constant_48 [Constant] outputs: [/proposal_generator/anchor_generator/Constant_48_output_0 -> ()[INT32]], 
[03/30/2026-15:48:20] [TRT] [V] Parsing node: /proposal_generator/anchor_generator/Mul_5 [Mul]
[03/30/2026-15:48:20] [TRT] [V] Searching for input: /proposal_generator/anchor_generator/Gather_4_output_0
[03/30/2026-15:48:20] [TRT] [V] Searching for input: /proposal_generator/anchor_generator/Constant_48_output_0
[03/30/2026-15:48:20] [TRT] [V] /proposal_generator/anchor_generator/Mul_5 [Mul] inputs: [/proposal_generator/anchor_generator/Gather_4_output_0 -> ()[INT32]], [/proposal_generator/anchor_generator/Constant_48_output_0 -> ()[INT32]], 
[03/30/2026-15:48:20] [TRT] [V] Registering layer: /proposal_generator/anchor_generator/Constant_48_output_0 for ONNX node: /proposal_generator/anchor_generator/Constant_48_output_0
[03

ERROR:EngineBuilder:Failed to load ONNX file: /home/alan_khang/dev/mirror3d/output/deploy/2026-03-30-15-47/converted.onnx
ERROR:EngineBuilder:In node 1395 (importSlice): UNSUPPORTED_NODE: Assertion failed: (axes.allValuesKnown()) && "This version of TensorRT does not support dynamic axes."


256